# Python 101 - Solutions
## Chapter IX

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_09.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
import random
import re

import requests
from bs4 import BeautifulSoup

USER_AGENTS = [
    'Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.111 Safari/537.36',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:15.0) Gecko/20100101 Firefox/15.0.1',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/42.0.2311.135 Safari/537.36 Edge/12.246',
    'Mozilla/5.0 (PlayStation 4 3.11) AppleWebKit/537.73 (KHTML, like Gecko)',
]


def get_header(agents):
    return {'User-agent': random.choice(agents)}

### I. SelectorGadget and the telex pictures

Nothing to solve - but worth pointing out that the notebook's own text is the lesson: the screenshots show a telex layout that no longer exists. `.placeholder_ img` still works today, and it will stop working too. **Selectors rot; the method does not.**

In [ ]:
soup = BeautifulSoup(requests.get('https://telex.hu',
                                  headers=get_header(USER_AGENTS)).content,
                     'html.parser')

images = [image.get('src') for image in soup.select('.placeholder_ img')]
print(len(images), 'images')
print(images[0])

assert len(images) > 10
assert all(url.startswith('http') for url in images if url)

### Exercise I: I want to be the very best...

Pokemon card listings on vatera. The listing links carry the `product_link` class - 50 of them on the first page.

In [ ]:
VATERA = 'https://www.vatera.hu/listings/index.php'

response = requests.get(VATERA, params={'q': 'pokemon kartya'},
                        headers=get_header(USER_AGENTS))
response.raise_for_status()
vatera = BeautifulSoup(response.content, 'html.parser')

card_urls = [link.get('href') for link in vatera.select('a.product_link')]
# the same item is linked from its picture and its title, so de-duplicate
card_urls = list(dict.fromkeys(card_urls))

print(len(card_urls), 'card listings on the first page')
for url in card_urls[:5]:
    print('  ', url[:88])

assert len(card_urls) > 10
assert all(url.startswith('http') for url in card_urls)

### Exercise II: Like no one ever was

Amazon's markup is a thicket of `a-` classes, but the useful handles are stable enough:

| what | selector |
|---|---|
| one result | `div[data-component-type="s-search-result"]` |
| title | `h2` |
| price | `.a-price .a-offscreen` |
| rating | `.a-icon-alt` |
| delivery | `div[data-cy="delivery-recipe"]` |

The exercise insists you **store** the results rather than print them - so a list of dictionaries, which is exactly what `pandas` will want in two chapters' time.

In [ ]:
AMAZON = 'https://www.amazon.de/s'

response = requests.get(AMAZON, params={'k': 'pokemon cards', 'language': 'en_GB'},
                        headers=get_header(USER_AGENTS))
response.raise_for_status()
amazon = BeautifulSoup(response.content, 'html.parser')


def text_or_none(element):
    return element.get_text(' ', strip=True) if element else None


listings = []
for card in amazon.select('div[data-component-type="s-search-result"]'):
    rating = text_or_none(card.select_one('.a-icon-alt'))
    listings.append({
        'title': text_or_none(card.select_one('h2')),
        'price': text_or_none(card.select_one('.a-price .a-offscreen')),
        # '4.4 out of 5 stars' -> 4.4
        'score': float(rating.split()[0].replace(',', '.')) if rating else None,
        'delivery': text_or_none(card.select_one('div[data-cy="delivery-recipe"]')),
    })

print(len(listings), 'listings stored\n')
for item in listings[:4]:
    print(f"  {(item['title'] or '?')[:40]:42s} {str(item['price'])[:12]:14s} "
          f"{item['score']}  {(item['delivery'] or '-')[:26]}")

assert len(listings) > 20
assert any(item['price'] for item in listings)
assert any(item['score'] for item in listings)
assert all(item['score'] is None or 0 <= item['score'] <= 5 for item in listings)

### II. Selenium: the election results site

The solution is the demo in the notebook - the thing to make sure students actually notice is the **contrast**: `requests` gets ~2 400 characters of loading spinner, the browser renders the real numbers. That is the entire reason selenium exists.

Two practical notes for the room:
- The first `webdriver.Chrome()` call is slow because Selenium Manager is downloading the driver. It only happens once.
- Always `driver.quit()` when you are done, or you leave orphaned Chrome processes eating memory for the rest of the session.

In [ ]:
URL = 'https://vtr.valasztas.hu/ogy2026'

plain = requests.get(URL).text
print('requests gave us', len(plain), 'characters')
assert 'spinner' in plain or 'root' in plain
# the numbers are simply not in there
assert 'TISZA' not in plain
print("...and the word 'TISZA' does not appear in it at all.")

### III. Querying APIs

> **Left open on purpose.** The `bud.hu/api/ajaxFlights/` endpoint the notebook > uses now returns `404` - Budapest Airport rebuilt the site and moved the > flight data behind a lazily-loaded javascript chunk. The screenshots in > `pics/RESTAPI_*.png` show the old site too.
>
> The *method* in the notebook (Network tab → filter to `xhr` → Preview →
> Copy as cURL → curlconverter.com) is completely unchanged and still the right
> way to do this. Only the example target needs replacing.

A ready-made, cookie-free alternative if you want one: the election API behind the site above. `config.json` tells you which data snapshot to read, then every result file hangs off that.

In [ ]:
ELECTION = 'https://vtr.valasztas.hu/ogy2026/data'

config = requests.get(f'{ELECTION}/config.json',
                      headers=get_header(USER_AGENTS)).json()
print('config.json:', config)

results = requests.get(
    f"{ELECTION}/{config['szavossz']}/szavossz/SzervezetekEredmenye.json",
    headers=get_header(USER_AGENTS)).json()

print('\nkeys:', list(results.keys()))
print('rows:', len(results['list']))
print('first row:', {k: v for k, v in list(results['list'][0].items())[:5]})

assert 'list' in results and len(results['list']) > 5
assert config['szavossz']

### Exercise IV: Find the parameters yourself

**The trap first.** CheapShark answers python's default user agent with a `400` and a message asking for a descriptive one like `'MyApp/1.0 (contact@...)'`. Do exactly that and it *still* returns `400`. Only a browser-looking agent gets through - so `get_header(USER_AGENTS)` from the top of the notebook.

Worth spending two minutes on: the server's error message is simply wrong, and no amount of reading documentation would have told you. You find this by trying.

In [ ]:
API = 'https://www.cheapshark.com/api/1.0'

# what the students will hit first
naive = requests.get(f'{API}/deals')
print('no user agent      ->', naive.status_code, naive.json().get('error', '')[:60])

# what the error message asks for
polite = requests.get(f'{API}/deals', headers={'User-Agent': 'Python101/1.0 (me@example.com)'})
print('what it asked for  ->', polite.status_code)

# what actually works
working = requests.get(f'{API}/deals', headers=get_header(USER_AGENTS))
print('browser user agent ->', working.status_code)

assert naive.status_code == 400
assert polite.status_code == 400      # the message is lying
assert working.status_code == 200

In [ ]:
HEADERS = get_header(USER_AGENTS)

# 1. one parameter at a time
print('parameter exploration')
for params in ({}, {'pageSize': 5}, {'upperPrice': 5}, {'AAA': 1},
               {'storeID': 1}, {'sortBy': 'Metacritic'}):
    deals = requests.get(f'{API}/deals', params=params, headers=HEADERS).json()
    first = deals[0]['title'][:26] if deals else '-'
    print(f"  {str(params):30s} -> {len(deals):3d} rows, first: {first}")

# 2. storeID -> shop name
stores = {store['storeID']: store['storeName']
          for store in requests.get(f'{API}/stores', headers=HEADERS).json()}
print(f'\n{len(stores)} shops, e.g.', dict(list(stores.items())[:4]))

assert stores['1'] == 'Steam'
assert len(stores) > 10

In [ ]:
# 3. the 20 biggest discounts on games rated 80+ on metacritic
deals = requests.get(f'{API}/deals', headers=HEADERS, params={
    'metacritic': 80,
    'onSale': 1,
    'sortBy': 'Savings',
    'pageSize': 20,
}).json()

for deal in deals:
    print(f"{deal['title'][:34]:36s} was ${deal['normalPrice']:>6s}, "
          f"now ${deal['salePrice']:>6s} "
          f"(-{float(deal['savings']):.0f}%) @ {stores[deal['storeID']]}")

assert len(deals) == 20
assert all(int(d['metacriticScore']) >= 80 for d in deals)
assert all(float(d['salePrice']) <= float(d['normalPrice']) for d in deals)
# sorted by savings, descending
savings = [float(d['savings']) for d in deals]
assert savings == sorted(savings, reverse=True)

**Extra:** functionised, so the parameter names live in one place.

In [ ]:
def find_deals(min_metacritic=None, max_price=None, store=None,
               sort_by='Savings', on_sale=True, limit=10):
    """Ask CheapShark for deals. Returns a list of dicts."""
    params = {'pageSize': limit, 'sortBy': sort_by}
    if on_sale:
        params['onSale'] = 1
    if min_metacritic is not None:
        params['metacritic'] = min_metacritic
    if max_price is not None:
        params['upperPrice'] = max_price
    if store is not None:
        params['storeID'] = store

    response = requests.get(f'{API}/deals', params=params,
                            headers=get_header(USER_AGENTS))
    response.raise_for_status()
    return response.json()


cheap_and_good = find_deals(min_metacritic=85, max_price=10, limit=5)
for deal in cheap_and_good:
    print(f"{deal['title'][:34]:36s} ${deal['salePrice']:>6s}  "
          f"mc={deal['metacriticScore']}")

assert len(cheap_and_good) <= 5
assert all(float(d['salePrice']) <= 10 for d in cheap_and_good)
assert all(int(d['metacriticScore']) >= 85 for d in cheap_and_good)

### Exercise V: Vote counting

The modern election sites block scripted requests (`423 Locked`), but the 2018
results are still served as plain html tables.

**One trap, and it is a good one.** The page's html is malformed: the very first
`<tr>` of the table *wraps all the other rows*, so `find_all('td')` on it returns
all **636** cells at once and you end up with 107 constituencies instead of 106 -
the first one counted twice.

The fix is to check the **shape** of each row: a real data row has exactly 6
cells. Getting into the habit of checking that a row looks like you expect,
instead of assuming, is worth more than this particular answer.

The full national scrape is in `python101_10_scraping_vote_results.ipynb`.

In [ ]:
VOTE_BASE = 'http://valasztas.hu/dyn/pv18/szavossz/hu/'

overview = BeautifulSoup(
    requests.get(VOTE_BASE + 'oevker.html', headers=get_header(USER_AGENTS)).content,
    'html.parser')

rows = overview.find('table', {'border': '1'}).find_all('tr')
print('raw <tr> count:', len(rows))
print('cells in the first row:', len(rows[0].find_all('td')), '<- the wrapper row!')

regions = []
for row in rows:
    cells = row.find_all('td')
    # a real data row has exactly 6 cells; the wrapper row has 636
    if len(cells) != 6:
        continue
    link = cells[1].find('a')
    if link:
        regions.append((cells[0].get_text().strip(),
                        cells[2].get_text().strip(),
                        link.get('href')))

print(f'\n{len(regions)} constituencies found')
print(regions[0])

assert len(regions) == 106, len(regions)
assert len({href for _, _, href in regions}) == 106, 'no duplicates'

In [ ]:
# now one constituency in detail
county, name, sub_url = regions[0]

detail = BeautifulSoup(
    requests.get(VOTE_BASE + sub_url, headers=get_header(USER_AGENTS)).content,
    'html.parser')

# `string=` not `text=`: bs4 4.13 renamed that argument
table = detail.find(string='A szavazatok száma jelöltenként').findNext('table')

candidates = []
for row in table.find_all('tr')[1:]:
    cells = [cell.get_text().replace('\xa0', '').replace('%', '').strip()
             for cell in row.find_all('td')]
    if len(cells) > 3:
        candidates.append([county, name] + cells[:-1])

print(f'{county} - {name}: {len(candidates)} candidates\n')
for candidate in candidates[:5]:
    print('  ', candidate)

assert len(candidates) > 3
assert all(row[0] == county for row in candidates)